# Evaluation (development period, 2016-2023)

Runs the tests fixed in `docs/preregistration.md`.
The holdout (2024-2026) is not touched here; it lives in `04_holdout.ipynb` and is run once at the end.

Sign convention: Thesis A (informed shorting) predicts a **negative** IC, meaning high short volume leads to underperformance.

In [1]:
import numpy as np
import pandas as pd

from shortvol import evaluate as ev, features, finra, prices

pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:.4f}".format)

panel = pd.read_parquet(prices.ROOT / "data" / "processed" / "panel.parquet")
days = finra.trading_days("2016-01-01", "2026-08-31")
V0 = features.estimate_v0(features.add_all(panel, days))["v0"]  # 2016-2019 short volume only
df = features.add_all(panel, days, v0=V0)

SIGNALS = ["svr5", "svr1", "svr_abnormal", "svr5_shrunk"]
CONTROLS = ["ret1", "ret5", "log_adv20", "abnormal_volume"]
TARGETS = ["fwd_oc", "fwd_cc"]
for s in SIGNALS:
    df[s + "_n"] = ev.neutralize(df, s, CONTROLS)
df["size_tercile"] = ev.terciles(df, "log_adv20", ["small", "mid", "large"])
df["offex_tercile"] = ev.terciles(df, "off_exchange_share", ["low", "mid", "high"])

In [2]:
d = df[df.date <= "2023-12-29"]
print(f"development stock-days: {len(d):,}, days: {d.date.nunique()}")

development stock-days: 1,000,295, days: 2012


## 1. Headline: 4 signals x 2 targets, raw and neutralized
Raw IC is the signal as is.
Neutralized IC removes day-t return, 5-day return, size (20-day dollar volume), abnormal volume and industry.
The gap between them shows how much of the signal is reversal, size or liquidity in disguise.

In [3]:
rows = []
for s in SIGNALS:
    for t in TARGETS:
        for kind, col in [("raw", s), ("neutralized", s + "_n")]:
            rows.append({"signal": s, "target": t, "version": kind, **ev.summarize_ic(ev.daily_ic(d, col, t))})
headline = pd.DataFrame(rows).set_index(["signal", "target", "version"])
headline

mean_ic    nw_t  ic_ir_annual  pct_positive  days
signal       target version                                                       
svr5         fwd_oc raw           0.0019  1.1382        0.4186        0.5025  2008
                    neutralized   0.0003  0.2619        0.0976        0.5040  1998
             fwd_cc raw          -0.0012 -0.7509       -0.2705        0.4836  2008
                    neutralized  -0.0018 -1.3387       -0.4897        0.4805  1998
svr1         fwd_oc raw          -0.0037 -2.2912       -0.8519        0.4871  2012
                    neutralized  -0.0045 -3.4652       -1.2930        0.4720  1998
             fwd_cc raw          -0.0011 -0.7054       -0.2545        0.4901  2012
                    neutralized  -0.0011 -0.8901       -0.3228        0.4885  1998
svr_abnormal fwd_oc raw           0.0013  1.0244        0.3667        0.5114  1967
                    neutralized  -0.0002 -0.1740       -0.0637        0.4967  1967
             fwd_cc raw           0.0010  0.8741        0.3029        0.5165  1967
                    neutralized  -0.0002 -0.1495       -0.0518        0.5018  1967
svr5_shrunk  fwd_oc raw           0.0018  1.0676        0.3970        0.5003  1973
                    neutralized   0.0003  0.2581        0.0968        0.5068  1973
             fwd_cc raw          -0.0014 -0.8280       -0.3005        0.4881  1973
                    neutralized  -0.0019 -1.4867       -0.5457        0.4830  1973

## 2. Quintile long-short (primary target)
Long the top quintile of the signal, short the bottom quintile, equal weight, rebalanced daily.
Turnover is the daily sum of absolute weight changes (a full flip of both legs is 4).
Break-even is the one-way cost in bps that erases the gross spread. It is reported, not pass/fail.

In [4]:
pd.DataFrame({f"{s}_n": ev.quintile_spread(d, s + "_n", "fwd_oc") for s in SIGNALS}).T

,spread_bps,spread_nw_t,turnover,breakeven_bps
svr5_n,0.2041,0.2956,0.7672,0.2660
svr1_n,-1.7985,-2.8094,2.1115,0.8517
svr_abnormal_n,-0.1184,-0.2323,0.8850,0.1338
svr5_shrunk_n,0.2124,0.3145,0.7588,0.2799


## 3. Decay
IC of the primary neutralized signal against the open-to-close return 1, 2, 5, 10 and 20 days ahead.
Informed trading should decay slowly; price pressure and reversal should die within a day or two.

In [5]:
pd.DataFrame({k: ev.summarize_ic(ev.daily_ic(d.assign(y=ev.shift_target(d, "fwd_oc", k)), "svr5_n", "y"))
              for k in [1, 2, 5, 10, 20]}).T

,mean_ic,nw_t,ic_ir_annual,pct_positive,days
1,0.0003,0.2619,0.0976,0.5040,1998.0000
2,0.0026,1.9277,0.7231,0.5183,1997.0000
5,0.0027,2.0445,0.7482,0.5271,1994.0000
10,0.0018,1.4746,0.5219,0.5103,1989.0000
20,0.0016,1.2308,0.4536,0.5144,1979.0000


## 4. Splits (primary signal, primary target)
By size, by period, and by off-exchange share.
Thesis B predicts the relation weakens or flips where off-exchange share is high.

In [6]:
d = d.assign(period=pd.cut(d.date, pd.to_datetime(["2015-12-31", "2019-12-31", "2021-12-31", "2023-12-31"]),
                           labels=["2016-19", "2020-21", "2022-23"]))
pd.concat({
    "size": ev.ic_by_group(d, "svr5_n", "fwd_oc", "size_tercile"),
    "period": ev.ic_by_group(d, "svr5_n", "fwd_oc", "period"),
    "off-exchange share": ev.ic_by_group(d, "svr5_n", "fwd_oc", "offex_tercile"),
})

mean_ic    nw_t  ic_ir_annual  pct_positive      days
size               large     0.0006  0.2824        0.1009        0.5020 1998.0000
                   mid       0.0000  0.0235        0.0085        0.4965 1998.0000
                   small     0.0007  0.3451        0.1275        0.5095 1998.0000
period             2016-19   0.0004  0.1973        0.1050        0.5030  992.0000
                   2020-21  -0.0002 -0.0774       -0.0580        0.5069  505.0000
                   2022-23   0.0009  0.3068        0.2240        0.5030  501.0000
off-exchange share high     -0.0049 -2.3331       -0.8316        0.4885 1998.0000
                   low       0.0024  1.3121        0.4626        0.5065 1998.0000
                   mid       0.0034  1.7059        0.6227        0.5135 1998.0000

## 5. Success bar (development part)
1. Mean IC negative with |NW t| >= 3.
2. The same sign in at least 2 of the 3 development periods.
Criterion 3 (holdout) is checked in `04_holdout.ipynb`.

In [7]:
primary = headline.loc[("svr5", "fwd_oc", "neutralized")]
periods = ev.ic_by_group(d, "svr5_n", "fwd_oc", "period")
c1 = primary.mean_ic < 0 and abs(primary.nw_t) >= 3
c2 = (np.sign(periods.mean_ic) == np.sign(primary.mean_ic)).sum() >= 2
print(f"1. mean IC {primary.mean_ic:.4f}, NW t {primary.nw_t:.2f}: {'PASS' if c1 else 'FAIL'}")
print(f"2. same sign in {(np.sign(periods.mean_ic) == np.sign(primary.mean_ic)).sum()} of 3 periods: {'PASS' if c2 else 'FAIL'}")

1. mean IC 0.0003, NW t 0.26: FAIL
2. same sign in 2 of 3 periods: PASS


## Appendix: pre-registered follow-up
The pre-registration said that if the primary signal failed after neutralization, the next test is the slow version: 20-day SVR against the following 5-day return, where the published evidence sits.
Target here: adjusted close t+1 to close t+6 (the first five sessions after the file is published).

In [8]:
thin = df.total_volume < features.FLOOR
sv, tv = df.short_volume.mask(thin), df.total_volume.mask(thin)
g = df.groupby("ticker")
ok20 = g.day.diff(19) == 19
df["svr20"] = (sv.groupby(df.ticker).transform(lambda s: s.rolling(20).sum())
               / tv.groupby(df.ticker).transform(lambda s: s.rolling(20).sum())).where(ok20)
ok6 = (g.day.shift(-6) - df.day == 6) & (g.day.shift(-1) - df.day == 1)
df["fwd_5d"] = (g.adj_close.shift(-6) / g.adj_close.shift(-1) - 1).where(ok6)
df["svr20_n"] = ev.neutralize(df, "svr20", CONTROLS)
dev = df[df.date <= "2023-12-29"]
pd.DataFrame({v: ev.summarize_ic(ev.daily_ic(dev, c, "fwd_5d")) for v, c in [("raw", "svr20"), ("neutralized", "svr20_n")]}).T

,mean_ic,nw_t,ic_ir_annual,pct_positive,days
raw,-0.0100,-2.9465,-1.9567,0.4501,1993.0000
neutralized,-0.0083,-3.2936,-2.1536,0.4486,1993.0000
